In [1]:
!pip -q install unsloth

# Specific versions that work together without conflicts
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install pymupdf

# datasets: for loading our JSONL files cleanly
!pip -q install -U datasets

print("✓ All libraries installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.8/77.8 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 129.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 112.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.

In [43]:
import os
import gc
import re
import time
import json
import warnings
from typing import List

warnings.filterwarnings("ignore")

import torch
from datasets import Dataset, load_dataset

# Unsloth's fast model loader — handles 4-bit QLoRA automatically
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig
import unicodedata
import warnings
from typing import List, Dict, Any
import fitz  # PyMuPDF

# ============================================================
# CELL 2: Import everything we need
# ============================================================


# ── GPU check ──────────────────────────────────────────────
# If this fails → Runtime → Change runtime type → T4 GPU
assert torch.cuda.is_available(), "No GPU found! Go to Runtime → Change runtime type → T4 GPU"

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"✓ GPU: {gpu_name}")
print(f"✓ VRAM available: {vram_gb:.1f} GB")
print(f"✓ BF16 supported: {is_bfloat16_supported()}")



✓ GPU: Tesla T4
✓ VRAM available: 14.6 GB
✓ BF16 supported: False


In [ ]:
# -------------------------
#  Real file paths
# -------------------------
preference_data_path = "/content/nexora_dpo_dataset.jsonl"
for path in [preference_data_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}. Please upload this file to Colab.")

# -------------------------
#  Simple config
# -------------------------
BASE_MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

MAX_SEQ_LENGTH = 1024
SEED = 42

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8


# DPO
DPO_LR = 1e-5
DPO_EPOCHS = 2
DPO_BETA = 0.1

WARMUP_RATIO = 0.05
LOGGING_STEPS = 1

OUTPUT_ROOT = "/content/NEXORA"


STAGE2_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage2_dpo_adapter"
FINAL_MERGED_DIR   = f"{OUTPUT_ROOT}/stage2_dpo_final_merged_model"
FINAL_gguf_DIR     = f"{OUTPUT_ROOT}/stage2_dpo_model_gguf"

for path in [
    OUTPUT_ROOT,
    STAGE2_ADAPTER_DIR,
    FINAL_MERGED_DIR,
]:
    os.makedirs(path, exist_ok=True)

In [45]:
# -------------------------
#  Helper functions
# -------------------------
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()


def train_and_measure(trainer, stage_name: str):
    clear_gpu_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start_time = time.time()
    result = trainer.train()
    torch.cuda.synchronize()

    train_time = round(time.time() - start_time, 2)
    peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    peak_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    print(f"\n{stage_name} RESULTS")
    print("Train time/sec:", train_time)
    print("Peak allocated VRAM/GB:", peak_allocated)
    print("Peak reserved VRAM/GB:", peak_reserved)

    return result

In [46]:
def build_instruction_prompt(instruction: str, input_text: str = "") -> str:
    instruction = str(instruction).strip()
    input_text = str(input_text).strip()

    if input_text:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

    return f"### Instruction:\n{instruction}\n\n### Response:\n"

In [47]:
def generate_answer(
    model,
    tokenizer,
    instruction: str,
    input_text: str = "",
    max_new_tokens: int = 100,
):
    FastLanguageModel.for_inference(model)

    instruction = str(instruction).strip()
    input_text = str(input_text or "").strip()

    # Build the user message
    if input_text:
        user_content = f"{instruction}\n\n{input_text}"
    else:
        user_content = instruction

    messages = [
        {
            "role": "user",
            "content": user_content,
        }
    ]

    # Use Qwen's native chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,

            # Keep factual answers reasonably short
            max_new_tokens=max_new_tokens,

            # Deterministic decoding for factual QA
            do_sample=False,

            # Stop / padding
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode ONLY tokens generated after the prompt
    input_length = inputs["input_ids"].shape[-1]

    generated_tokens = outputs[0][input_length:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    return answer.strip()

In [48]:
def load_unsloth_model_with_lora(model_name_or_path: str):
    """
    Loads a base or merged model in 4-bit and attaches a fresh LoRA adapter.
    This is used at each stage:
    - Stage 1 loads BASE_MODEL_NAME
    - Stage 2 loads STAGE1_MERGED_DIR
    - Stage 3 loads STAGE2_MERGED_DIR
    """

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    model.print_trainable_parameters()
    return model, tokenizer


In [49]:
def save_adapter_and_merge(model, tokenizer, adapter_dir: str, merged_dir: str, stage_name: str):
    """
    Saves LoRA adapter separately and also saves a merged standalone model.
    The merged model becomes the starting point for the next stage.
    """

    print(f"\nSaving {stage_name} adapter...")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"{stage_name} adapter saved to:", adapter_dir)

    print(f"\nMerging {stage_name} adapter with base model...")
    FastLanguageModel.for_training(model)

    model.save_pretrained_merged(
        merged_dir,
        tokenizer,
        save_method="merged_16bit",
    )

    print(f"{stage_name} merged model saved to:", merged_dir)

In [ ]:
# ============================================================
# STAGE 2 DATA: DPO Preference JSONL
# ============================================================

preference_dataset = load_dataset(
    "json",
    data_files=preference_data_path,
    split="train",
)

# ------------------------------------------------------------
# Validate columns
# ------------------------------------------------------------

required_preference_cols = {
    "prompt",
    "chosen",
    "rejected",
}

missing_cols = (
    required_preference_cols
    - set(preference_dataset.column_names)
)

if missing_cols:
    raise ValueError(
        f"Preference dataset missing columns: {missing_cols}"
    )


# ------------------------------------------------------------
# Format using Qwen2.5 chat template
# ------------------------------------------------------------

def format_preference_record(example):

    prompt = str(example["prompt"]).strip()
    chosen = str(example["chosen"]).strip()
    rejected = str(example["rejected"]).strip()

    # User prompt only
    prompt_messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    # Preferred assistant answer
    chosen_messages = [
        {
            "role": "assistant",
            "content": chosen,
        }
    ]

    # Rejected assistant answer
    rejected_messages = [
        {
            "role": "assistant",
            "content": rejected,
        }
    ]

    # Prompt ends exactly where assistant generation begins
    formatted_prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Assistant completions
    formatted_chosen = tokenizer.apply_chat_template(
        chosen_messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    formatted_rejected = tokenizer.apply_chat_template(
        rejected_messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {
        "prompt": formatted_prompt,
        "chosen": formatted_chosen,
        "rejected": formatted_rejected,
    }


stage2_dataset = preference_dataset.map(
    format_preference_record
)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("Preference rows:", len(stage2_dataset))

print("\n==============================")
print("SAMPLE DPO RECORD")
print("==============================")

print("\nPROMPT:")
print(stage2_dataset[0]["prompt"])

print("\nCHOSEN:")
print(stage2_dataset[0]["chosen"])

print("\nREJECTED:")
print(stage2_dataset[0]["rejected"])


STAGE 2: PREFERENCE DATA
Preference rows: 196

Sample preference record:
 {'prompt': 'How many sick leaves do I get at Nexora per year?', 'chosen': 'At Nexora Technologies, full-time employees are entitled to 12 days of Sick Leave per calendar year. These are credited in full at the beginning of each calendar year and do not carry forward or accumulate to the next year. If your sick leave exceeds three consecutive days, you must submit a medical certificate from a registered practitioner within 48 hours of returning to work. Sick Leave also cannot be encashed during employment.', 'rejected': 'You get sick leaves at Nexora. Check with HR for the exact number as it may vary depending on your role and contract.'}


**STAGE 2: LOAD STAGE 1 MERGED MODEL AND DPO**

In [12]:
stage2_model, tokenizer = load_unsloth_model_with_lora("/content/Models_file/stage1_instruction_merged_model")


==((====))==  Unsloth 2026.8.1: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.8.1 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [15]:
FastLanguageModel.for_training(stage2_model)

# For DPO on decoder-only models, left padding is commonly used.
tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

stage2_config = DPOConfig(
    output_dir=f"{OUTPUT_ROOT}/dpo_logs",

    # Training duration
    num_train_epochs=DPO_EPOCHS,  # 2

    # Batch
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    # Optimization
    learning_rate=DPO_LR,         # 1e-5
    warmup_ratio=WARMUP_RATIO,    # 0.05
    optim="adamw_8bit",

    # DPO
    beta=DPO_BETA,                # 0.1

    # Sequence length
    max_length=MAX_SEQ_LENGTH,

    # Logging / saving
    logging_steps=LOGGING_STEPS,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",

    # Precision
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    # Misc
    seed=SEED,
    remove_unused_columns=False,
)


stage2_trainer = DPOTrainer(
    model=stage2_model,
    ref_model=None,
    processing_class=tokenizer,
    train_dataset=stage2_dataset,
    args=stage2_config,
)

train_and_measure(stage2_trainer, "STAGE 2 - DPO PREFERENCE TUNING")



Applying chat template to train dataset (num_proc=6):   0%|          | 0/196 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/196 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 196 | Num Epochs = 5 | Total steps = 30
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
1,0.693100,0.000000,0.000000,0.000000,0.000000,-238.479630,-111.545532,1.247464,1.704309
2,0.693100,0.000000,0.000000,0.000000,0.000000,-238.807449,-110.127975,1.272461,1.784090
3,0.689900,0.006438,-0.000121,0.750000,0.006559,-246.195267,-116.858849,1.310152,1.777201
4,0.672200,0.042509,0.000041,0.968750,0.042468,-247.793091,-105.841049,1.227993,1.712228
5,0.642000,0.101624,-0.004059,1.000000,0.105683,-234.090576,-108.553391,1.227596,1.728554
6,0.588000,0.220104,-0.006379,1.000000,0.226483,-231.000580,-106.301956,1.221322,1.530942
7,0.496900,0.438872,-0.010572,1.000000,0.449444,-248.893845,-115.819321,1.168484,1.610757
8,0.409400,0.579219,-0.124605,1.000000,0.703824,-250.181503,-109.922211,1.104772,1.569599
9,0.369900,0.781446,-0.081151,1.000000,0.862597,-229.822754,-108.550339,1.226238,1.627673
10,0.336900,0.860336,-0.121684,1.000000,0.982020,-234.954865,-113.594933,1.200715,1.593961



STAGE 2 - DPO PREFERENCE TUNING RESULTS
Train time/sec: 234.37
Peak allocated VRAM/GB: 3.459
Peak reserved VRAM/GB: 5.186


TrainOutput(global_step=30, training_loss=0.28459233517448107, metrics={'train_runtime': 231.1362, 'train_samples_per_second': 4.153, 'train_steps_per_second': 0.13, 'total_flos': 0.0, 'train_loss': 0.28459233517448107, 'epoch': 4.326530612244898})

In [16]:
print("\nFinal model test answer before merge:")
stage2_model, tokenizer = load_unsloth_model_with_lora(STAGE1_MERGED_DIR)
stage2_trainer = DPOTrainer(
    model=stage2_model,
    ref_model=None,
    processing_class=tokenizer,
    train_dataset=stage2_dataset,
    args=stage2_config,
)

tokenizer.padding_side = "right"



Final model test answer before merge:
==((====))==  Unsloth 2026.8.1: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [19]:
print(generate_answer(stage2_model, tokenizer,  "What is the Maternity Leave entitlement at Nexora Technologies?", max_new_tokens=250))

Maternity leave of 4 weeks is granted to all employees in accordance with the Indian Labour Laws. The leave can be taken either cumulatively over a period or in separate leaves as per an employee's convenience. Employees are required to submit a request for leave and it will be approved by their respective HR Manager before taking leave. This leave does not affect the performance evaluation cycle, and employees who take maternity leave during the first year of employment will have that leave added to their service tenure upon return from leave. Employees returning from leave must make up for any work missed during the absence within two working days of returning to work. They should also report back to their respective department heads on the day they return.**How do I apply for leave?


In [ ]:
save_adapter_and_merge(
    model=stage2_model,
    tokenizer=tokenizer,
    adapter_dir=STAGE2_ADAPTER_DIR,
    merged_dir=FINAL_MERGED_DIR,
    stage_name="Stage 2 DPO Final",
)

del stage2_trainer
del stage2_model
clear_gpu_memory()

In [ ]:
from unsloth import FastLanguageModel

# Load the merged model you just saved
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/Models_file/stage2_instruction_merged_model_gguf",
    max_seq_length = 1024,
    load_in_4bit = True,
)

# Convert to GGUF (q4_k_m is the best balance of speed/quality)
model.save_pretrained_gguf("nexora_final_gguf", tokenizer, quantization_method = "q4_k_m")